# Letterboxd Review Scraper - Google Colab

This notebook allows you to search and fetch reviews from Letterboxd films.

## How to use:
1. Click **Runtime** → **Run all** (or run cells individually)
2. The first cell installs dependencies
3. The second cell contains the scraper code
4. The third cell is where you customize your search

---

## Step 1: Install Dependencies

Run this cell first to install required packages.

In [ ]:
!pip install -q requests beautifulsoup4 lxml
print("✅ Dependencies installed!")

## Step 2: Scraper Code

This cell contains all the scraper logic. Just run it once.

In [ ]:
import requests
from bs4 import BeautifulSoup
import re
import time
from typing import List, Dict, Optional

class LetterboxdReviewScraper:
    """Scraper for Letterboxd film reviews"""

    BASE_URL = "https://letterboxd.com"

    def __init__(self):
        self.session = requests.Session()
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
            'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
            'Accept-Language': 'en-US,en;q=0.5',
            'Accept-Encoding': 'gzip, deflate, br',
            'DNT': '1',
            'Connection': 'keep-alive',
            'Upgrade-Insecure-Requests': '1',
            'Sec-Fetch-Dest': 'document',
            'Sec-Fetch-Mode': 'navigate',
            'Sec-Fetch-Site': 'none',
            'Cache-Control': 'max-age=0'
        })

    def extract_film_slug(self, url: str) -> Optional[str]:
        """Extract film slug from Letterboxd URL"""
        if url.startswith('http'):
            match = re.search(r'letterboxd\.com/film/([^/]+)', url)
            if match:
                return match.group(1)
        else:
            return url.strip('/')
        return None

    def get_reviews(self, film_slug: str, page: int = 1, sort: str = 'popular') -> List[Dict]:
        """Fetch reviews for a film"""
        ajax_url = f"{self.BASE_URL}/film/{film_slug}/reviews/by/{sort}/page/{page}/"
        time.sleep(0.5)

        try:
            ajax_headers = {
                'X-Requested-With': 'XMLHttpRequest',
                'Referer': f'{self.BASE_URL}/film/{film_slug}/',
            }
            response = self.session.get(ajax_url, headers=ajax_headers, timeout=15)
            response.raise_for_status()
        except requests.RequestException:
            try:
                response = self.session.get(ajax_url, timeout=15)
                response.raise_for_status()
            except requests.RequestException as e:
                print(f"❌ Error fetching reviews: {e}")
                print(f"URL: {ajax_url}")
                return []

        soup = BeautifulSoup(response.content, 'lxml')
        reviews = []
        review_items = soup.find_all('li', class_='film-detail')

        for item in review_items:
            review = self._parse_review_item(item)
            if review:
                reviews.append(review)

        return reviews

    def _parse_review_item(self, item) -> Optional[Dict]:
        """Parse a single review item"""
        review = {}

        user_link = item.find('strong', class_='name')
        if user_link:
            review['username'] = user_link.get_text(strip=True)

        rating_span = item.find('span', class_='rating')
        if rating_span:
            stars = len(rating_span.find_all('span', class_='rated-'))
            review['rating'] = stars / 2
        else:
            review['rating'] = None

        review_body = item.find('div', class_='body-text')
        if review_body:
            for span in review_body.find_all('span', class_='collapsed-text'):
                span.decompose()
            review_text = review_body.get_text(separator=' ', strip=True)
            review['text'] = review_text
        else:
            review['text'] = ""

        like_link = item.find('a', class_='has-icon icon-liked icon-16 has-count')
        if like_link:
            like_count = like_link.get_text(strip=True)
            try:
                review['likes'] = int(like_count) if like_count else 0
            except ValueError:
                review['likes'] = 0
        else:
            review['likes'] = 0

        date_span = item.find('span', class_='_nobr')
        if date_span:
            review['date'] = date_span.get_text(strip=True)

        return review if review else None

    def get_film_info(self, film_slug: str) -> Optional[Dict]:
        """Get basic film information"""
        url = f"{self.BASE_URL}/film/{film_slug}/"
        time.sleep(0.5)

        try:
            response = self.session.get(url, timeout=15)
            response.raise_for_status()
        except requests.RequestException as e:
            print(f"❌ Error fetching film info: {e}")
            return None

        soup = BeautifulSoup(response.content, 'lxml')
        info = {}

        title_tag = soup.find('h1', class_='headline-1')
        if title_tag:
            info['title'] = title_tag.get_text(strip=True)

        year_tag = soup.find('small', class_='number')
        if year_tag:
            info['year'] = year_tag.get_text(strip=True)

        director_tag = soup.find('span', class_='prettify')
        if director_tag:
            info['director'] = director_tag.get_text(strip=True)

        rating_meta = soup.find('meta', {'name': 'twitter:data2'})
        if rating_meta:
            info['average_rating'] = rating_meta.get('content', '')

        return info

def print_review(review: Dict, index: int):
    """Pretty print a review"""
    print(f"\n{'='*80}")
    print(f"Review #{index}")
    print(f"{'='*80}")
    print(f"👤 User: {review.get('username', 'Anonymous')}")

    if review.get('rating'):
        stars = '⭐' * int(review['rating']) + '☆' * (5 - int(review['rating']))
        print(f"⭐ Rating: {stars} ({review['rating']}/5)")
    else:
        print("⭐ Rating: No rating")

    if review.get('date'):
        print(f"📅 Date: {review['date']}")

    if review.get('likes'):
        print(f"❤️  Likes: {review['likes']}")

    if review.get('text'):
        print(f"\n{review['text']}")
    else:
        print("\n(No review text)")

print("✅ Scraper code loaded!")

## Step 3: Search for Reviews

### Customize your search here:

**FILM_URL**: Change this to any Letterboxd film URL or slug
- Example: `https://letterboxd.com/film/taxi-1998/`
- Or just: `taxi-1998`

**PAGE**: Which page of reviews to fetch (default: 1)

**SORT**: How to sort reviews
- `'popular'` - Most liked reviews (default)
- `'recent'` - Most recent reviews
- `'highest-rated'` - Highest rated reviews
- `'lowest-rated'` - Lowest rated reviews

In [ ]:
# ⚙️ CUSTOMIZE YOUR SEARCH HERE ⚙️

FILM_URL = "https://letterboxd.com/film/taxi-1998/"  # Change this!
PAGE = 1                                               # Change page number
SORT = "popular"                                       # Change sort: 'popular', 'recent', 'highest-rated', 'lowest-rated'

# =====================================
# Don't change below this line
# =====================================

scraper = LetterboxdReviewScraper()

# Extract film slug
film_slug = scraper.extract_film_slug(FILM_URL)
if not film_slug:
    print(f"❌ Invalid film URL or slug: {FILM_URL}")
else:
    print(f"🎬 Fetching film information...")
    film_info = scraper.get_film_info(film_slug)

    if film_info:
        print(f"\n{'='*80}")
        print(f"🎥 Film: {film_info.get('title', 'Unknown')}")
        if film_info.get('year'):
            print(f"📅 Year: {film_info['year']}")
        if film_info.get('director'):
            print(f"🎬 Director: {film_info['director']}")
        if film_info.get('average_rating'):
            print(f"⭐ Average Rating: {film_info['average_rating']}")
        print(f"{'='*80}")

    # Get reviews
    print(f"\n📝 Fetching reviews (page {PAGE}, sorted by {SORT})...\n")
    reviews = scraper.get_reviews(film_slug, page=PAGE, sort=SORT)

    if not reviews:
        print("❌ No reviews found.")
    else:
        print(f"✅ Found {len(reviews)} reviews:\n")

        for i, review in enumerate(reviews, 1):
            print_review(review, i)

        print(f"\n{'='*80}")
        print(f"📊 Total reviews displayed: {len(reviews)}")
        print(f"{'='*80}\n")

## Optional: Export Reviews to CSV

Run this cell to save reviews to a CSV file that you can download.

In [ ]:
import csv
from google.colab import files

if 'reviews' in locals() and reviews:
    filename = f"{film_slug}_reviews_page{PAGE}.csv"
    
    with open(filename, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=['username', 'rating', 'date', 'likes', 'text'])
        writer.writeheader()
        writer.writerows(reviews)
    
    print(f"✅ Exported {len(reviews)} reviews to {filename}")
    print("⬇️  Downloading file...")
    files.download(filename)
else:
    print("❌ No reviews to export. Run the search cell first!")

## Tips

- **Change the film**: Edit `FILM_URL` in Step 3
- **Get more reviews**: Increase `PAGE` number (2, 3, 4...)
- **Sort differently**: Change `SORT` to see different reviews
- **Download data**: Run the CSV export cell to save reviews
- **Troubleshooting**: If you get errors, try running again in a few minutes

## More Examples

Try these films:
- `https://letterboxd.com/film/parasite-2019/`
- `https://letterboxd.com/film/the-shawshank-redemption/`
- `https://letterboxd.com/film/pulp-fiction/`
- `https://letterboxd.com/film/spirited-away/`